In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from pathlib import Path

In [2]:
META_DIR = Path('..') / 'meta_files'
GDRIVE   = Path('/Users/andrewleduc/Library/CloudStorage/GoogleDrive-research@slavovlab.net/My Drive/MS/Users/aleduc/AAS_rev')

PSM_path   = GDRIVE / 'psm.tsv'          # updated: full TMT quantification
fasta_path = META_DIR / 'output_MTP_cl_fp.fasta'

# ── Age annotation ──────────────────────────────────────────────────────────
# TODO: map TMT channels to Young / Old once experiment annotation is confirmed.
channel_age = {
    '126'  : 'Young',
    '127N' : 'Young',
    '127C' : 'Young',
    '128N' : 'Young',
    '128C' : 'Young',
    '129N' : 'Old',
    '129C' : 'Old',
    '130N' : 'Old',
    '130C' : 'Old',
    '131N' : 'Old',
}

## Build SAAP → BP mapping from PSM (1-AA matching)

In [ ]:
# Parse fasta: collect all SAAP sequences (new_peptide and MTP_ entries)
saap_sequences = set()
current_is_saap = False
with open(fasta_path) as f:
    for line in f:
        line = line.strip()
        if line.startswith('>'):
            parts = line.split('|')
            id_field = parts[2].split()[0] if len(parts) >= 3 else ''
            current_is_saap = ('new_peptide' in id_field) or id_field.startswith('MTP_')
        elif current_is_saap and line:
            saap_sequences.add(line)

print(f'Unique SAAP sequences in fasta (new_peptide + MTP): {len(saap_sequences)}')

# Load PSM — include Intensity (precursor MS1) for RAAS scaling
TMT_COLS = [f'Intensity mouse_brain_1_{ch}'
            for ch in ['126','127N','127C','128N','128C','129N','129C','130N','130C','131N']]
usecols = ['Peptide', 'Protein', 'Intensity'] + TMT_COLS

psm = pd.read_csv(PSM_path, sep='\t', usecols=usecols)
psm[TMT_COLS] = psm[TMT_COLS].apply(pd.to_numeric, errors='coerce')
psm['Intensity'] = pd.to_numeric(psm['Intensity'], errors='coerce')
print(f'Total PSM rows: {len(psm)}')

# Split into SAAP rows and canonical rows
is_saap_row = psm['Protein'].str.contains('new_peptide|MTP_', na=False)
saap_psm = psm[is_saap_row & psm['Peptide'].isin(saap_sequences)].copy()
canonical_psm = psm[~is_saap_row].copy()
print(f'SAAP PSM rows: {len(saap_psm)}, unique SAAPs: {saap_psm["Peptide"].nunique()}')

# Build 1-AA Hamming matching: for each SAAP find canonical peptide differing by exactly 1 AA
canonical_by_len = defaultdict(set)
for seq in canonical_psm['Peptide'].unique():
    canonical_by_len[len(seq)].add(seq)

def find_bp(saap):
    candidates = [c for c in canonical_by_len.get(len(saap), [])
                  if sum(a != b for a, b in zip(saap, c)) == 1]
    return candidates[0] if candidates else None

bp_map = {s: find_bp(s) for s in saap_psm['Peptide'].unique()}
matched   = sum(1 for v in bp_map.values() if v is not None)
unmatched = sum(1 for v in bp_map.values() if v is None)
print(f'SAAPs with 1-AA matched BP: {matched}  |  no match: {unmatched}')

saap_psm['BP'] = saap_psm['Peptide'].map(bp_map)
saap_psm = saap_psm.dropna(subset=['BP'])

bp_sequences = set(saap_psm['BP'])
bp_psm = canonical_psm[canonical_psm['Peptide'].isin(bp_sequences)].copy()
print(f'BP PSM rows: {len(bp_psm)}, unique BPs: {bp_psm["Peptide"].nunique()}')

Unique SAAP sequences in fasta (new_peptide + MTP): 5489


## Load PSM data and compute RAAS

In [ ]:
# PSM loading and SAAP/BP filtering are done in the cell above

In [ ]:
# RAAS per channel = (TMT_SAAP_ch * Precursor_SAAP) / (TMT_BP_ch * Precursor_BP)
#
# TMT reporters give relative channel distribution within one peptide.
# Multiplying by the precursor MS1 intensity scales across peptides (SAAP vs BP),
# so the ratio captures true substitution abundance per sample.

channels = ['126','127N','127C','128N','128C','129N','129C','130N','130C','131N']

# Sum TMT and precursor intensities per peptide across all PSMs
saap_sum = saap_psm.groupby(['Peptide','BP'])[['Intensity'] + TMT_COLS].sum().reset_index()
bp_sum   = (bp_psm.groupby('Peptide')[['Intensity'] + TMT_COLS]
                   .sum().reset_index()
                   .rename(columns={'Peptide': 'BP', 'Intensity': 'Intensity_bp',
                                    **{f'Intensity mouse_brain_1_{ch}': f'TMT_bp_{ch}'
                                       for ch in channels}}))

merged = saap_sum.merge(bp_sum, on='BP')
merged = merged.rename(columns={'Intensity': 'Intensity_saap',
                                 **{f'Intensity mouse_brain_1_{ch}': f'TMT_saap_{ch}'
                                    for ch in channels}})
print(f'SAAP/BP pairs after merge: {len(merged)}')

raas_rows = []
for _, row in merged.iterrows():
    prec_saap = row['Intensity_saap']
    prec_bp   = row['Intensity_bp']
    if not (prec_saap > 0 and prec_bp > 0):
        continue
    for ch in channels:
        tmt_saap = row[f'TMT_saap_{ch}']
        tmt_bp   = row[f'TMT_bp_{ch}']
        if tmt_saap > 0 and tmt_bp > 0:
            raas = (tmt_saap * prec_saap) / (tmt_bp * prec_bp)
            raas_rows.append({
                'SAAP'   : row['Peptide'],
                'BP'     : row['BP'],
                'channel': ch,
                'RAAS'   : raas,
                'age'    : channel_age[ch],
            })

raas_df = pd.DataFrame(raas_rows)
print(f'RAAS observations: {len(raas_df)}, unique SAAPs: {raas_df["SAAP"].nunique()}')

## RAAS by age group

In [ ]:
raas_df['log2_RAAS'] = np.log2(raas_df['RAAS'])

mean_per_saap_age = (raas_df.groupby(['SAAP','age'])['log2_RAAS']
                              .mean().reset_index())

age_order = ['Young', 'Old']
age_colors = {'Young': '#4C72B0', 'Old': '#DD8452'}

fig, ax = plt.subplots(figsize=(5, 5))
sns.swarmplot(data=mean_per_saap_age, x='age', y='log2_RAAS',
              order=age_order, palette=age_colors, size=4, alpha=0.7, ax=ax)

for age in age_order:
    med = mean_per_saap_age.loc[mean_per_saap_age['age'] == age, 'log2_RAAS'].median()
    xi = age_order.index(age)
    ax.hlines(med, xi - 0.35, xi + 0.35, colors='black', linewidths=2)

ax.set_xlabel('Age group', fontsize=12)
ax.set_ylabel('log2(RAAS)', fontsize=12)
ax.set_title('RAAS by age group (Mouse Brain)', fontsize=13)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
n_young = mean_per_saap_age[mean_per_saap_age['age']=='Young']['SAAP'].nunique()
n_old   = mean_per_saap_age[mean_per_saap_age['age']=='Old']['SAAP'].nunique()
ax.text(0, ax.get_ylim()[0], f'n={n_young}', ha='center', va='bottom', fontsize=9)
ax.text(1, ax.get_ylim()[0], f'n={n_old}',   ha='center', va='bottom', fontsize=9)
sns.despine()
plt.tight_layout()
# plt.savefig('mouse_brain_RAAS_by_age.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Normalised RAAS: subtract per-SAAP mean across channels
saap_mean_all = raas_df.groupby('SAAP')['log2_RAAS'].mean().rename('saap_mean')
raas_norm = raas_df.merge(saap_mean_all, on='SAAP')
raas_norm['log2_RAAS_norm'] = raas_norm['log2_RAAS'] - raas_norm['saap_mean']

mean_norm_per_saap_age = (raas_norm.groupby(['SAAP','age'])['log2_RAAS_norm']
                                    .mean().reset_index())

# Clip to 1st-99th percentile
lo, hi = np.percentile(mean_norm_per_saap_age['log2_RAAS_norm'].dropna(), [1, 99])
mean_norm_per_saap_age['log2_RAAS_norm'] = mean_norm_per_saap_age['log2_RAAS_norm'].clip(lo, hi)

fig, ax = plt.subplots(figsize=(5, 5))
sns.boxplot(data=mean_norm_per_saap_age, x='age', y='log2_RAAS_norm',
            order=age_order, palette=age_colors, width=0.5,
            flierprops=dict(marker='.', markersize=3, alpha=0.4), ax=ax)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_xlabel('Age group', fontsize=12)
ax.set_ylabel('Normalised log2(RAAS)', fontsize=12)
ax.set_title('Normalised RAAS by age group (Mouse Brain)', fontsize=13)
sns.despine()
plt.tight_layout()
# plt.savefig('mouse_brain_RAAS_norm_by_age.pdf', bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats

raas_df['log2_RAAS'] = np.log2(raas_df['RAAS'])

young_channels = [ch for ch, age in channel_age.items() if age == 'Young']
old_channels   = [ch for ch, age in channel_age.items() if age == 'Old']

# Pivot: rows = SAAP, columns = channel
raas_pivot = raas_df.pivot_table(index='SAAP', columns='channel',
                                  values='log2_RAAS', aggfunc='mean')

y_cols = [c for c in young_channels if c in raas_pivot.columns]
o_cols = [c for c in old_channels   if c in raas_pivot.columns]

# Intersection: SAAPs with at least 1 value in each group
mask = raas_pivot[y_cols].notna().any(axis=1) & \
       raas_pivot[o_cols].notna().any(axis=1)
raas_pivot = raas_pivot[mask]
print(f'SAAPs in both Young and Old (intersection): {len(raas_pivot)}')

# Welch t-test per SAAP, using only non-NaN channels per row
results = []
for saap, row in raas_pivot.iterrows():
    y_vals = row[y_cols].dropna().values.astype(float)
    o_vals = row[o_cols].dropna().values.astype(float)
    fc = o_vals.mean() - y_vals.mean()   # log2 space → log2 fold change
    if len(y_vals) >= 2 and len(o_vals) >= 2:
        t, p = stats.ttest_ind(o_vals, y_vals, equal_var=False)
    else:
        p = np.nan
    results.append({'SAAP': saap, 'log2_FC': fc, 'pval': p})

volcano = pd.DataFrame(results).dropna(subset=['pval'])
volcano['neg_log10_p'] = -np.log10(volcano['pval'].clip(lower=1e-300))
print(f'SAAPs tested (>=2 replicates per group): {len(volcano)}')

# ── Plot ──────────────────────────────────────────────────────────────────────
sig_thresh = 0.05
fc_thresh  = 1.0   # log2(2) = 1 → 2-fold change

volcano['sig'] = (volcano['pval'] < sig_thresh) & (volcano['log2_FC'].abs() > fc_thresh)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(volcano.loc[~volcano['sig'], 'log2_FC'],
           volcano.loc[~volcano['sig'], 'neg_log10_p'],
           c='#8C8C8C', s=20, alpha=0.6, edgecolors='none', label='n.s.')
ax.scatter(volcano.loc[volcano['sig'], 'log2_FC'],
           volcano.loc[volcano['sig'], 'neg_log10_p'],
           c='#E64B35', s=25, alpha=0.9, edgecolors='none',
           label=f'p<{sig_thresh}, |FC|>2×')

ax.axhline(-np.log10(sig_thresh), color='black', linestyle='--', linewidth=0.8)
ax.axvline( fc_thresh, color='grey', linestyle='--', linewidth=0.6)
ax.axvline(-fc_thresh, color='grey', linestyle='--', linewidth=0.6)
ax.axvline(0, color='black', linewidth=0.5)

ax.set_xlabel('log2(RAAS) Old − Young', fontsize=12)
ax.set_ylabel('−log10(p-value)', fontsize=12)
ax.set_title('RAAS volcano: Old vs Young (Mouse Brain)', fontsize=13)
ax.legend(fontsize=9)
sns.despine()
plt.tight_layout()
# plt.savefig('mouse_brain_volcano_old_vs_young.pdf', bbox_inches='tight')
plt.show()

n_up   = ((volcano['sig']) & (volcano['log2_FC'] > 0)).sum()
n_down = ((volcano['sig']) & (volcano['log2_FC'] < 0)).sum()
print(f'Significant: {n_up} higher in Old, {n_down} higher in Young')